# Эмбеддинги CLIP для всех кадров BDD100K (Colab, T4)

Что нужно в `MyDrive/bdd100k/`:
- `bdd100k_images_100k.zip`;
- папка `check_cpu/` - эталон, посчитанный локально на CPU (`python -m src.embed --mode crop --limit 256 --device cpu --out artifacts/check_cpu`
  и то же с `--mode pad`): шесть файлов `clip_b32_{crop,pad}.npy`, `_names.parquet`, `_meta.json`.

Сначала те же 256 кадров считаются на GPU и сверяются с эталоном: если минимальный косинус не выше 0,999, ноутбук
останавливается. Потом полный проход по 100K кадров в двух вариантах (crop и pad). Он пишет на Drive куски
по 10 000 кадров: если сессия оборвётся, перезапустите все ячейки по порядку, готовые куски пропустятся.
Результат - `MyDrive/bdd100k/emb/`, локально его нужно положить в `artifacts/emb/` (папки `*_chunks` не нужны).

In [ ]:
import os
import subprocess

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/bdd100k"
REPO_URL = "https://github.com/artyombelyaev/road-frame-search.git"
REPO = "/content/road-frame-search"
ZIP = "/content/bdd100k_images_100k.zip"

In [ ]:
if not os.path.exists(REPO):
    !git clone -q $REPO_URL $REPO
else:
    !git -C $REPO pull -q
%cd $REPO
!git log --oneline -1
!pip install -q transformers==5.21.0
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))"

In [ ]:
# Читать 100K файлов с примонтированного Drive очень долго, поэтому архив копирую на диск Colab.
# Большие файлы Drive отдаёт с обрывами ("Operation canceled"), поэтому копирую с докачкой:
# после обрыва продолжаю с того байта, где остановился. Готово, только если совпали размер и md5.
import hashlib
import time
import zipfile

SRC = f"{DRIVE_DIR}/bdd100k_images_100k.zip"
PART = ZIP + ".part"
EXPECTED_MD5 = "c7a1d4db9af5a4691d5f6fee2a62e132"  # md5 архива, с которого считался эталон check_cpu


def md5(path, block=64 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while chunk := f.read(block):
            h.update(chunk)
    return h.hexdigest()


def copy_resumable(src, dst, block=64 << 20, retries=30):
    total = os.path.getsize(src)
    for attempt in range(retries):
        done = os.path.getsize(dst) if os.path.exists(dst) else 0
        if done == total:
            return
        if done > total:
            os.remove(dst)
            continue
        try:
            with open(src, "rb") as fi, open(dst, "ab") as fo:
                fi.seek(done)
                while chunk := fi.read(block):
                    fo.write(chunk)
        except OSError as e:
            print(f"обрыв на {os.path.getsize(dst) / 1e9:.2f} ГБ из {total / 1e9:.2f}: {e}. Повтор {attempt + 1}", flush=True)
            time.sleep(10)
    raise RuntimeError("не удалось скопировать архив с Drive")


print("на Drive:", os.path.getsize(SRC), "байт")
if not os.path.exists(ZIP):
    copy_resumable(SRC, PART)
    os.rename(PART, ZIP)
digest = md5(ZIP)
if digest != EXPECTED_MD5:
    # неполную копию следующий запуск докачает, полную, но испорченную - скачает заново
    if os.path.getsize(ZIP) < os.path.getsize(SRC):
        os.rename(ZIP, PART)
    else:
        os.remove(ZIP)
    raise RuntimeError(f"md5 {digest} не совпал с {EXPECTED_MD5}: запустите ячейку ещё раз")
with zipfile.ZipFile(ZIP) as z:
    n = sum(m.endswith(".jpg") for m in z.namelist())
assert n == 100_000, n
print("копия:", os.path.getsize(ZIP), "байт, md5 совпал, 100 000 кадров")

## 1. Сверка с CPU на 256 кадрах

In [ ]:
# вывод скрипта печатаю построчно: без этого Colab не показывает ни прогресс, ни текст ошибки
def run(*args):
    cmd = ["python", "-m", "src.embed", "--images-zip", ZIP, *args]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise RuntimeError(f"src.embed завершился с кодом {proc.returncode}")


for mode in ["crop", "pad"]:
    run("--mode", mode, "--limit", "256", "--out", "/content/check", "--compare-with", f"{DRIVE_DIR}/check_cpu")

## 2. Полный проход

In [ ]:
run("--mode", "crop", "--out", f"{DRIVE_DIR}/emb", "--workers", "2")

In [ ]:
run("--mode", "pad", "--out", f"{DRIVE_DIR}/emb", "--workers", "2")

In [ ]:
!ls -lh $DRIVE_DIR/emb
!cat $DRIVE_DIR/emb/clip_b32_crop_meta.json